# Fine-Tuning a Small Language Model with PEFT to Generate Midjourney Prompts

User-approved portable replacement for the historical Mistral/AutoTrain workflow: official SmolLM2-135M-Instruct, Transformers and PEFT LoRA on CPU. Uses all 117 MIT-licensed source rows for three epochs. Training-set fit is not a held-out quality benchmark, and this does not validate Mistral-7B. Full-precision CPU weights replace CUDA-specific 4-bit loading. No account, API key or upload is needed. Source blocks/transcripts remain verbatim in the nonexecuting appendix. Dataset copyright/license: Josh Bickett, MIT; see the included source license file.

Run cells in order. Dependencies and hardware requirements are listed below.

In [ ]:
import os, io, json, hashlib, tempfile, gc, math
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from huggingface_hub import snapshot_download
# python -m pip install torch==2.10.0 transformers==4.57.6 peft==0.21.0 accelerate==1.12.0 pandas==2.3.3 safetensors==0.8.0
torch.set_num_threads(4)
torch.manual_seed(22)
device = 'cpu'
print('Local CPU LoRA; no AutoTrain, paid API or Hub upload.')

In [ ]:
DATA_REVISION = '9606720fe6aaf540d214bce08ca4b21c578d6274'
DATA_URL = f'https://raw.githubusercontent.com/joshbickett/finetune-llama-2/{DATA_REVISION}/train.csv'
with urlopen(DATA_URL, timeout=60) as response:
    csv_bytes = response.read()
assert hashlib.sha256(csv_bytes).hexdigest() == 'abd8f1aa15d045741355452906b0a24025caefeb055ba06f6755b4a40b6d4707'
df = pd.read_csv(io.BytesIO(csv_bytes))
assert list(df.columns) == ['Concept', 'Funny Description Prompt', 'text'] and len(df) == 117
assert df.notna().all().all()
print('Source dataset rows:', len(df), '; missing values: 0')
print(df.head().to_string(index=False))

In [ ]:
print(df['text'].iloc[1])
print(df['text'].iloc[2])

In [ ]:
MODEL_ID = 'HuggingFaceTB/SmolLM2-135M-Instruct'
REVISION = '12fd25f77366fa6b3b4b768ec3050bf629380bac'
checkpoint = snapshot_download(MODEL_ID, revision=REVISION, token=False,
    cache_dir=os.environ.get('TDB_MODEL_CACHE'), allow_patterns=['*.json','*.safetensors','*.txt'])
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32,
    attn_implementation='eager', local_files_only=True, trust_remote_code=False).to(device)
model.config.use_cache = False
print('Model:', MODEL_ID, '; revision:', REVISION)
weight = Path(checkpoint)/'model.safetensors'
print('Model SHA256:', hashlib.sha256(weight.read_bytes()).hexdigest())

In [ ]:
# Convert each original Concept/response pair into this model's chat format.
# Every row is used, and no tokenized example is truncated.
texts = [tokenizer.apply_chat_template([
    {'role':'user','content': 'generate a midjourney prompt for '+row['Concept']},
    {'role':'assistant','content':row['Funny Description Prompt']}], tokenize=False)
    for _,row in df.iterrows()]
encoded = [tokenizer(text, add_special_tokens=False, return_tensors='pt') for text in texts]
lengths = [item['input_ids'].shape[1] for item in encoded]
assert len(encoded) == 117 and max(lengths) <= model.config.max_position_embeddings
probe = encoded[0]
def loss_on(item):
    return model(**item, labels=item['input_ids']).loss
def eval_loss():
    model.eval()
    numerator = denominator = 0
    with torch.no_grad():
        for item in encoded:
            count = item['input_ids'].numel()-1
            numerator += float(loss_on(item))*count; denominator += count
    return numerator/denominator
def frozen_hash(model):
    h = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        if 'lora_' not in name:
            h.update(name.encode());h.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
print('All rows retained:',len(encoded),'; min/max token lengths:',min(lengths),max(lengths))

In [ ]:
config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    lora_dropout=0.05, target_modules=['q_proj','v_proj'])
model = get_peft_model(model, config)
trainable = {name:p for name,p in model.named_parameters() if p.requires_grad}
assert trainable and all('lora_' in name for name in trainable)
targets = [module for module in model.modules() if hasattr(module,'lora_A')]
expected = sum(config.r*(module.in_features+module.out_features) for module in targets)
assert sum(p.numel() for p in trainable.values()) == expected
frozen_before = frozen_hash(model)
before = {name:p.detach().clone() for name,p in trainable.items()}
initial = eval_loss()
print('LoRA target matrices:',len(targets),'; trainable parameters:',expected)
print('Initial full-dataset training loss:',round(initial,6))

In [ ]:
optimizer = torch.optim.AdamW(list(trainable.values()), lr=2e-4, weight_decay=0.0)
EPOCHS, EFFECTIVE_BATCH = 3, 12
seen = [0]*len(encoded); nonzero = set(); steps = 0
generator = torch.Generator().manual_seed(22)
for epoch in range(EPOCHS):
    order = torch.randperm(len(encoded), generator=generator).tolist()
    model.train()
    for start in range(0,len(order),EFFECTIVE_BATCH):
        indices = order[start:start+EFFECTIVE_BATCH]
        optimizer.zero_grad(set_to_none=True)
        tokens = sum(encoded[index]['input_ids'].numel()-1 for index in indices)
        for index in indices:
            item = encoded[index]; out = model(**item, labels=item['input_ids'])
            independent = F.cross_entropy(out.logits[:,:-1,:].reshape(-1,out.logits.shape[-1]),
                item['input_ids'][:,1:].reshape(-1))
            torch.testing.assert_close(out.loss,independent,atol=1e-5,rtol=1e-5)
            assert torch.isfinite(out.loss)
            (out.loss*((item['input_ids'].numel()-1)/tokens)).backward()
            seen[index] += 1
        for name,p in trainable.items():
            assert p.grad is not None and torch.isfinite(p.grad).all()
            if p.grad.abs().max() > 0:nonzero.add(name)
        assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
        torch.nn.utils.clip_grad_norm_(list(trainable.values()),1.0)
        optimizer.step();steps += 1
    print('Completed epoch:',epoch+1)
final = eval_loss()
assert seen == [3]*117 and steps == 30
assert nonzero == set(trainable) and frozen_hash(model) == frozen_before
assert any(not torch.equal(before[name],p) for name,p in trainable.items())
assert math.isfinite(final) and final < initial
print(json.dumps({'rows':117,'epochs':3,'row_visits':sum(seen),'updates':steps,
    'initial_training_loss':round(initial,6),'final_training_loss':round(final,6),
    'frozen_base_unchanged':True,'all_adapter_gradients_finite_nonzero':True},sort_keys=True))

In [ ]:
artifacts = tempfile.TemporaryDirectory(prefix='tdb-prompt-adapter-')
adapter_path = Path(artifacts.name)/'adapter'
model.eval()
with torch.no_grad(): expected_logits = model(**probe).logits.detach().clone()
model.save_pretrained(adapter_path, safe_serialization=True)
tokenizer.save_pretrained(adapter_path)
restored_base = AutoModelForCausalLM.from_pretrained(checkpoint,dtype=torch.float32,
    attn_implementation='eager',local_files_only=True,trust_remote_code=False)
restored = PeftModel.from_pretrained(restored_base,adapter_path,
    device_map={'':'cpu'},torch_device='cpu').eval()
with torch.no_grad(): actual_logits = restored(**probe).logits
torch.testing.assert_close(actual_logits,expected_logits,atol=1e-5,rtol=1e-5)
print('Local adapter saved and reloaded; logits match.')

In [ ]:
prompt = 'generate a midjourney prompt for A person walks in the rain'
messages = [{'role':'user','content':prompt}]
inputs = tokenizer.apply_chat_template(messages,add_generation_prompt=True,return_tensors='pt',return_dict=True)
torch.manual_seed(22)
with torch.no_grad(): generated = restored.generate(**inputs,max_new_tokens=200,do_sample=True,
    temperature=0.7,top_p=0.9,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
new_tokens = generated[:,inputs['input_ids'].shape[1]:]
assert 0 < new_tokens.shape[1] <= 200
assert torch.equal(generated[:,:inputs['input_ids'].shape[1]],inputs['input_ids'])
print(tokenizer.decode(new_tokens[0],skip_special_tokens=True))
print('New tokens:',new_tokens.shape[1])

In [ ]:
merged = restored.merge_and_unload().eval()
with torch.no_grad(): merged_logits = merged(**probe).logits
torch.testing.assert_close(merged_logits,expected_logits,atol=2e-4,rtol=2e-4)
print('Merged adapter logits match; no Hub upload performed.')
del merged,restored,restored_base,model,optimizer,expected_logits,actual_logits,merged_logits
gc.collect()
artifacts.cleanup()

## Original archive appendix — not executable

These verbatim source blocks and transcripts preserve the historical examples. They are not the maintained workflow above.

### Original code 1

```text
!pip install pandas autotrain-advanced -q
```

### Original code 2

```text
!autotrain setup --update-torch
```

### Original code 3

```text
from huggingface_hub import notebook_login
```

### Original code 4

```text
!git clone https://github.com/joshbickett/finetune-llama-2.git
%cd finetune-llama-2
%mv train.csv ../train.csv
%cd ..
```

### Original code 5

```text
import pandas as pd
df = pd.read_csv("train.csv")
df
```

### Original code 6

```text
df['text'][1]
```

### Original code 7

```text
df['text'][2]
```

### Original code 8

```text
!autotrain llm --train --project_name mistral-7b-mj-finetuned \
```

### Original code 9

```text
!pip install -q peft  accelerate bitsandbytes safetensors
```

### Original code 10

```text
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
adapters_name = "youssefhosni/mistral-7b-mj-finetuned"
model_name = "bn22/Mistral-7B-Instruct-v0.1-sharded" #"mistralai/Mistral-7B-Instruct-v0.1"

device = "cuda" # the device to load the model onto
```

### Original code 11

```text
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
```

### Original code 12

```text
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    device_map='auto'
)
```

### Original code 13

```text
model = PeftModel.from_pretrained(model, adapters_name)
```

### Original code 14

```text
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.bos_token_id = 1

stop_token_ids = [0]

print(f"Successfully loaded the model {model_name} into memory")
```

### Original code 15

```text
text = "[INST] generate a midjourney prompt for A person walks in the rain [/INST]"

encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False)
model_input = encoded
model.to(device)
generated_ids = model.generate(**model_input, max_new_tokens=200, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])
```

### Original transcript 1

###Human: Generate a midjourney prompt for A robot on a first date

### Original transcript 2

###Assistant: A robot, with a bouquet of USB cables, nervously adjusting its antennas, at a romantic restaurant that serves electricity shots.

### Original transcript 3

###Human: Generate a midjourney prompt for A snail at a speed contest

### Original transcript 4

###Assistant: A snail, with a mini rocket booster, confidently lining up at the start line, with a crowd of cheering insects.

### Original transcript 5

[INST] generate a midjourney prompt for A person walks in the rain [/INST] “As you wander through the pouring rain, you can’t help but wonder what the world would be like if things were different. What if the rain was a symbol of the turmoil in your life, and the sunshine promised a brighter future? What if you suddenly found yourself lost in a small town where time stood still, and the people were trapped in a time loop? As you struggle to find your way back to reality, you discover a mysterious stranger who seems to hold the key to unlocking the secrets of the town and your own past.”</s>